# DIR-SkyFinder Baselines on Colab

Trains ResNet-50 / ViT-B/16 on the SkyFinder image → temperature task. All training logic lives in `dir_skyfinder/baseline.py` — this notebook just imports and calls it.

**Setup:**
1. Upload the entire `DIR_Code/` project to your Google Drive (default location: `MyDrive/DIR_Code/`).
2. Runtime → Change runtime type → GPU (T4 / V100 / A100).
3. Run cells top-to-bottom. Smoke test first to confirm setup; then the full runs.

In [1]:
# Mount Drive, set project path, import baseline, rebind paths to Drive copy.
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJ_PATH = "/content/drive/MyDrive/DIR_Code"  # change if your folder is elsewhere
sys.path.insert(0, PROJ_PATH)

import dir_skyfinder.baseline as b
b.PROJ = Path(PROJ_PATH)
b.DATA = b.PROJ / "data"
b.LABELS = b.DATA / "labels_with_images.csv"
b.SPLITS = b.DATA / "splits" / "loco_3fold.json"
b.IMG_DIR = b.DATA / "images"
b.RESULTS_DIR = b.PROJ / "results"

for p in [b.LABELS, b.SPLITS, b.IMG_DIR]:
    print(f"{'✓' if p.exists() else '✗'}  {p}")

Mounted at /content/drive
✗  /content/drive/MyDrive/DIR_Code/data/labels_with_images.csv
✗  /content/drive/MyDrive/DIR_Code/data/splits/loco_3fold.json
✓  /content/drive/MyDrive/DIR_Code/data/images


## (One-time) Download dataset to Drive

**Skip this cell if the path check above showed ✓ for all three** (`labels_with_images.csv`, `splits/loco_3fold.json`, `images/`).

First run on a fresh Drive: this downloads the SkyFinder dataset directly onto your Drive (faster than uploading from your laptop).

- Metadata CSV — ~92 MB, ~30 sec
- ~81k JPEGs across 47 cameras — ~3 GB, ~30–60 min on Colab network (host rate-limits to a few connections per IP)
- Produces `labels.parquet`, `labels_with_images.csv`, `splits/loco_3fold.json`

Image download is resumable — if Colab disconnects, just re-run this cell.

In [2]:
import os
os.chdir(PROJ_PATH)

# 1) Metadata CSV (~92 MB)
from urllib.request import urlretrieve
from pathlib import Path
csv = Path("data/complete_table_with_mcr.csv")
if not csv.exists():
    Path("data").mkdir(exist_ok=True)
    print("downloading metadata CSV…")
    urlretrieve("https://cs.valdosta.edu/~rpmihail/skyfinder/analysis/complete_table_with_mcr.csv", csv)
print(f"csv: {csv.stat().st_size:,} bytes")

# 2) Drop NaN/sentinel TempM -> labels.parquet
!python data/prep_labels.py

# 3) Download ~81k images (~30–60 min, resumable)
!python data/download_images.py

# 4) Filter labels to rows whose image is on disk -> labels_with_images.csv
!python data/filter_to_images.py

# 5) Build 3-fold leave-one-camera-out splits -> splits/loco_3fold.json
!python data/splits.py

# Re-check the bound paths in module b (they were defined in cell 1 already)
for p in [b.LABELS, b.SPLITS, b.IMG_DIR]:
    print(f"{'✓' if p.exists() else '✗'}  {p}")

csv: 92,389,332 bytes
[drop] 94,803 -> 94,468 rows  (335 dropped, 0.35%)
[TempM] min=-27.2  max=50.0  mean=14.10
[cams] 53 unique
[saved] data/labels.parquet  (937,890 bytes)
[plan] 53 cameras, 94,803 total rows
[cam 65] 3252 rows
cam 65: 100% 3252/3252 [00:10<00:00, 300.96img/s, err=0, ok=0, skip=3252] 0:10<00:01, 715.45img/s, err=0, ok=0, skip=2070]
[cam 75] 1829 rows
cam 75: 100% 1829/1829 [00:06<00:00, 304.78img/s, err=0, ok=0, skip=1829] 
[cam 162] 3246 rows
cam 162: 100% 3246/3246 [00:15<00:00, 207.01img/s, err=0, ok=136, skip=3110]10% 320/3246 [00:10<00:37, 77.20img/s, err=0, ok=15, skip=342]
[cam 204] 2494 rows
cam 204: 100% 2494/2494 [01:22<00:00, 30.19img/s, err=0, ok=2494, skip=0]
[cam 260] 3225 rows
cam 260: 100% 3225/3225 [02:17<00:00, 23.43img/s, err=0, ok=3225, skip=0]
[cam 623] 3741 rows
cam 623: 100% 3741/3741 [02:41<00:00, 23.11img/s, err=0, ok=3741, skip=0]
[cam 684] 4091 rows
cam 684: 100% 4091/4091 [02:48<00:00, 24.22img/s, err=0, ok=4091, skip=0]
[cam 858] 200 row

In [4]:
import os
os.chdir(PROJ_PATH)

# 1) Metadata CSV (~92 MB)
from urllib.request import urlretrieve
from pathlib import Path
csv = Path("data/complete_table_with_mcr.csv")
if not csv.exists():
    Path("data").mkdir(exist_ok=True)
    print("downloading metadata CSV…")
    urlretrieve("https://cs.valdosta.edu/~rpmihail/skyfinder/analysis/complete_table_with_mcr.csv", csv)
print(f"csv: {csv.stat().st_size:,} bytes")

# 2) Drop NaN/sentinel TempM -> labels.parquet
!python data/prep_labels.py

# 3) Download ~81k images (~30–60 min, resumable)
!python data/download_images.py

# 4) Filter labels to rows whose image is on disk -> labels_with_images.csv
!python data/filter_to_images.py

# 5) Build 3-fold leave-one-camera-out splits -> splits/loco_3fold.json
!python data/splits.py

# Re-check the bound paths in module b (they were defined in cell 1 already)
for p in [b.LABELS, b.SPLITS, b.IMG_DIR]:
    print(f"{'✓' if p.exists() else '✗'}  {p}")

csv: 92,389,332 bytes
[drop] 94,803 -> 94,468 rows  (335 dropped, 0.35%)
[TempM] min=-27.2  max=50.0  mean=14.10
[cams] 53 unique
[saved] data/labels.parquet  (937,890 bytes)
[plan] 53 cameras, 94,803 total rows
[cam 65] 3252 rows
cam 65: 100% 3252/3252 [00:11<00:00, 290.66img/s, err=0, ok=0, skip=3252] 
[cam 75] 1829 rows
cam 75: 100% 1829/1829 [00:08<00:00, 210.13img/s, err=0, ok=0, skip=1829]
[cam 162] 3246 rows
cam 162: 100% 3246/3246 [00:15<00:00, 212.04img/s, err=0, ok=0, skip=3246]
[cam 204] 2494 rows
cam 204: 100% 2494/2494 [00:17<00:00, 139.91img/s, err=0, ok=0, skip=2494]
[cam 260] 3225 rows
cam 260: 100% 3225/3225 [00:09<00:00, 326.06img/s, err=0, ok=0, skip=3225] 
[cam 623] 3741 rows
cam 623: 100% 3741/3741 [00:12<00:00, 294.52img/s, err=0, ok=0, skip=3741] 
[cam 684] 4091 rows
cam 684: 100% 4091/4091 [00:15<00:00, 258.90img/s, err=0, ok=0, skip=4091] 
[cam 858] 200 rows
cam 858: 100% 200/200 [00:00<00:00, 279.20img/s, err=0, ok=0, skip=200]
[cam 861] 945 rows
cam 861: 100%

In [2]:
# Device check
import torch
print("device:", b.get_device())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

device: cuda
gpu: NVIDIA L4
memory: 23.7 GB


## Smoke test (~2 min on T4)

1 epoch on a tiny subset, just to confirm everything wires up. Reads from Drive directly — fine for 2000 files; would be slow for full runs.

In [5]:
results = b.run_baseline(
    model="resnet50",
    epochs=1,
    train_subset=2000,
    val_subset=500,
    run_name="smoke_colab",
)

[env] device=cuda  run=smoke_colab
[cfg] {'model': 'resnet50', 'fold': 0, 'epochs': 1, 'batch_size': 32, 'lr': 0.001, 'num_workers': 2, 'train_subset': 2000, 'val_subset': 500, 'seed': 0, 'run_name': 'smoke_colab'}
[data] train=2,000  val=500
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 210MB/s]


KeyboardInterrupt: 

## Speed up: copy images to local Colab storage

Reading images over the Drive filesystem is slow (~50–100 ms per file random access). For full 20-epoch runs, copy once to `/content/` (Colab's local SSD) — takes a few minutes, then each epoch is ~10× faster.

Skip this section if you're only running smoke tests.

In [6]:
# rsync is idempotent: re-running just copies what's new.
import subprocess
LOCAL_IMG = Path("/content/skyfinder_images")
LOCAL_IMG.mkdir(exist_ok=True)
print("copying images from Drive to local SSD… (~3–5 min)")
subprocess.run(["rsync", "-a", str(b.IMG_DIR) + "/", str(LOCAL_IMG) + "/"], check=True)
n = sum(1 for _ in LOCAL_IMG.rglob("*.jpg"))
print(f"done: {n:,} images on local SSD")

# Rebind the module's IMG_DIR to the local copy
b.IMG_DIR = LOCAL_IMG
print("b.IMG_DIR is now", b.IMG_DIR)

copying images from Drive to local SSD… (~3–5 min)


KeyboardInterrupt: 

In [ ]:
# !tar -czf /content/drive/MyDrive/datasets/skyfinder_images.tar.gz -C /content/drive/MyDrive/datasets skyfinder_images

In [ ]:
from pathlib import Path
import subprocess
import time

ARCHIVE = Path("/content/drive/MyDrive/datasets/skyfinder_images.tar.gz")
LOCAL_ROOT = Path("/content/data")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

start = time.time()

print("Copying archive to local SSD...")
subprocess.run(
    ["cp", str(ARCHIVE), "/content/skyfinder_images.tar.gz"],
    check=True,
)

print("Unpacking archive locally...")
subprocess.run(
    ["tar", "-xzf", "/content/skyfinder_images.tar.gz", "-C", str(LOCAL_ROOT)],
    check=True,
)

elapsed = time.time() - start

b.IMG_DIR = LOCAL_ROOT / "skyfinder_images"

print("Done.")
print("b.IMG_DIR is now:", b.IMG_DIR)
print(f"Elapsed: {elapsed/60:.2f} min")

## Full baseline: ResNet-50

20 epochs on fold 0. T4 ETA ~30 min after local-copy speedup.

In [ ]:
results_resnet = b.run_baseline(
    model="resnet50",
    epochs=30,
    fold=0,
    batch_size=64,
    num_workers=4,
    run_name="baseline_resnet50_fold0",
)

## Full baseline: ViT-B/16

20 epochs on fold 0. T4 ETA ~90 min.

In [ ]:
results_vit = b.run_baseline(
    model="vit_b_16",
    epochs=30,
    fold=0,
    batch_size=64,
    num_workers=4,
    run_name="baseline_vit_fold0",
)

## Compare results

Load the saved JSONs and plot per-bin MAE + per-epoch convergence.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

resnet = b.load_results(b.RESULTS_DIR / "baseline_resnet50_fold0.json")
vit = b.load_results(b.RESULTS_DIR / "baseline_vit_fold0.json")

table = pd.DataFrame([resnet["final_val"], vit["final_val"]], index=["ResNet-50", "ViT-B/16"])
print(table.round(3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for r, name in [(resnet, "ResNet-50"), (vit, "ViT-B/16")]:
    h = pd.DataFrame(r["history"])
    ax1.plot(h["epoch"], h["val_mae"], marker="o", label=name)
ax1.set(xlabel="epoch", ylabel="val MAE (°C)", title="Convergence"); ax1.legend(); ax1.grid(alpha=0.3)

table.plot(kind="bar", ax=ax2, ylabel="MAE (°C)", title="Per-bin MAE (val)")
ax2.set_xticklabels(table.index, rotation=0)
plt.tight_layout(); plt.show()

## Reuse a saved baseline (no retraining)

Every `run_baseline` call writes two files to `RESULTS_DIR` on Drive (so they survive Colab session restarts):

- `<run_name>.json` — config + per-epoch history + per-bin MAE + val predictions (~few hundred KB)
- `<run_name>.pt` — `state_dict` of the **best-val-MAE epoch** (~100 MB for ResNet-50, ~340 MB for ViT-B/16)

The cell below lists what's saved, loads the ResNet baseline back into a fresh model, and verifies inference works by spot-checking MAE on 200 val samples. Use this pattern any time you want to (re)use a trained baseline — for inference, transfer, or later as a starting point for LDS/FDS experiments.

In [ ]:
import numpy as np

# What's on disk?
print("saved checkpoints:")
for p in sorted(b.RESULTS_DIR.glob("*.pt")):
    print(f"  {p.name:40s} {p.stat().st_size/1e6:>6.1f} MB")

# Load the ResNet baseline state_dict back into a fresh model
sd = b.load_checkpoint("baseline_resnet50_fold0")
net = b.make_model("resnet50")
net.load_state_dict(sd)
net.to(b.get_device()).eval()

# Spot-check: run on 200 val samples and compare to the reported best_val_mae
_, val_loader, _, _ = b.build_loaders(fold=0, val_subset=200, num_workers=2)
preds, ys = b.evaluate(net, val_loader, b.get_device())
print(f"\nreloaded ResNet-50 MAE on 200 val samples: {np.mean(np.abs(preds-ys)):.3f}")
print(f"(expected ~{resnet['best_val_mae']:.3f} from the saved JSON)")